In [ ]:
import numpy as np 
from pathlib import Path
import csv
import os
import glob
import keras
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Model
from tensorflow.keras.layers import Lambda
import seaborn as sns
from sklearn.metrics import confusion_matrix,classification_report
from tensorflow.keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from itertools import combinations
import time
from scipy.signal import fftconvolve
import re
import shutil
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['axes.unicode_minus']=False
import math
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

In [ ]:
def robust_read_csv(file_path,channels=7,length=96000):

    raw_lines = []
    total_count = 0
    bad_count = 0
    with open(file_path,'r') as f:
        for line in f:
            
            line = line.strip()
            
            fields = line.split(',')
            
            values = []
            for item in fields:
                total_count += 1
                try:
                    value = float(item)
                    if np.isnan(value) or np.isinf(value) or abs(value)>10:
                        value = 0.0
                        bad_count += 1
                    values.append(value)
                except:
                    values.append(0.0)
                    bad_count += 1
            
            if len(values)>length:
                values = values[:length]
            elif len(values)<length:
                values = values + [0]*(length-len(values))
            raw_lines.append(values)
    
    arr = np.zeros((channels,length),dtype = np.float32)
    n_lines = min(len(raw_lines),channels)
    for r in range(n_lines):
        arr[r,:] = raw_lines[r]
    
    arr_max = np.max(arr)
    arr_min = np.min(arr)
    arr_mean = np.mean(arr)
    bad_ratio = bad_count/total_count
    arr = arr[-4:,-9600:]
    return arr


def robust_read_rf_csv(file_path,length = 500000):
    samples = []
    total_exception_count = 0
    total_point_count = 0
    
    with open(file_path,'r') as f:
        for idx,line in enumerate(f):
            fields = line.strip().split(',')
            values = []
            for item in fields:
                try:
                    value = float(item)
                    values.append(value)
                except Exception:
                    values.append(0)
            
            if len(values)>length:
                values = values[:length]
            elif len(values)<length:
                values = values + [0]*(length-len(values))
            
            values = np.array(values,dtype = np.float32)
        
            values = values/(2**15)
            
            mask = np.isnan(values)|np.isinf(values)|(np.abs(values)>1)
            exception_count = np.sum(mask)
            total_exception_count += exception_count
            total_point_count += len(values)
            values[mask]=0.0

            v_max = np.max(values)
            v_min = np.min(values)
            v_mean = np.mean(values)
            exception_ratio = exception_count/len(mask)
            samples.append(values)
    arr = np.array(samples,dtype=np.float32)

    total_exception_ratio = total_exception_count/total_point_count
    arr_max = np.max(arr)
    arr_min = np.min(arr)
    arr_mean = np.mean(arr)
    print(f'[robust_read_rf_csv] {file_path}: max = {arr_max:.2f} min = {arr_min:.2f} mean = {arr_mean:.2f}')
    print(f'[robust_read_rf_csv] {file_path}: Total number of outliers={total_exception_count} Ratio={total_exception_ratio:.6f}')
    return arr
     

In [ ]:
# =================  Reading data and label ================
t0 = time.time()
T_audio = 0.2 
fs_audio = 48000 
N_audio = int(fs_audio*T_audio) 
num_classes = 4 
mic_num = 4 
dt_audio = 1/fs_audio 
df_audio = fs_audio/N_audio 
t_audio = np.arange(0,N_audio)*dt_audio 

data_folder = 'Experiment_dataset'
drone_types = ['Air3S','Mavic2pro','Mini3pro','Mini5pro']
RF_suffixes = ('RF_I.csv','RF_Q.csv')
loca_suffixes = ['x.csv','y.csv','z.csv']
seq_length = 500000
height_folders = ['1H','2H']
def extract_num(filename):
    m = re.search(r'audio_(\d+)\.csv',filename)
    return int(m.group(1)) if m else -1
    

def load_microphone_signal(
    data_folder,
    drone_types,
    fs_audio,
    N_audio,
    RF_suffixes,
    loca_suffixes,
    num_classes = 4,
    mic_num = 4,
    height_folders = ['1H','2H']
    ):


    print('Reading data')
    all_audio = []
    all_labels = []
    all_RF = []
    all_positions = []
    
    
    for idx,drone_type in enumerate(drone_types):
        label = np.zeros(num_classes)
        label[idx] = 1

        type_folder = os.path.join(data_folder,drone_type)
        for height in height_folders:
            height_path = os.path.join(type_folder,height)
            # read passed index
            passed_index_file = os.path.join(height_path,'passed_indices.csv')
            passed_indices = pd.read_csv(passed_index_file)["passed_index"].to_numpy()
            passed_indices = passed_indices - 1
            # read location label
            x_pos = pd.read_csv(os.path.join(height_path,loca_suffixes[0]),header=None).values.squeeze()
            y_pos = pd.read_csv(os.path.join(height_path,loca_suffixes[1]),header=None).values.squeeze()
            z_pos = pd.read_csv(os.path.join(height_path,loca_suffixes[2]),header=None).values.squeeze()
            
            x_pos = x_pos[passed_indices]
            y_pos = y_pos[passed_indices]
            z_pos = z_pos[passed_indices]
            position_labels = np.stack([x_pos,y_pos,z_pos],axis=1) # (num_samples,3)
        
            print(position_labels.shape)
            all_positions.append(position_labels)
            
            file_list = [f for f in os.listdir(height_path) if f.startswith('audio_') and f.endswith('.csv')]
            file_list = sorted(file_list,key=extract_num)
            selected_files = [file_list[i] for i in passed_indices]
            print(selected_files[:10])
            for file in selected_files:
                file_path = os.path.join(height_path,file)
                try:
                    arr = robust_read_csv(file_path)
                    
                    all_audio.append(arr)
                    all_labels.append(label)
                except Exception as e:
                    print(f'{file_path} Loading failed, error message reported:{e}')
                    continue
            
            rf_I_file = os.path.join(height_path,"RF_I.csv")
            rf_Q_file = os.path.join(height_path,"RF_Q.csv")
            if not (os.path.exists(rf_I_file) and os.path.exists(rf_Q_file)):
                print(f"{rf_I_file},{rf_Q_file} not exist,skip")
                continue
            try:
                rf_I_data = robust_read_rf_csv(rf_I_file,length=500_000)
                rf_Q_data = robust_read_rf_csv(rf_Q_file,length=500_000)
                if rf_I_data.shape != rf_Q_data.shape:
                    print(f"{height_path} The number of samples of RF_I and RF_Q is inconsistent, skip")
                    continue
                
                rf_I_data = rf_I_data[passed_indices]
                rf_Q_data = rf_Q_data[passed_indices]

                rf_data = np.stack([rf_I_data,rf_Q_data],axis=1)
                all_RF.append(rf_data)
                print(f"{drone_type}-{height} remain RF samples:{rf_data.shape[0]}")
            except Exception as e:
                print(f'{height_path} RF_I/Q loading failed, error message reported:{e}')
                continue

    all_audio = np.asarray(all_audio)
    all_labels = np.asarray(all_labels)
    all_positions = np.vstack(all_positions)
    all_RF = np.vstack(all_RF)


    X_audio = all_audio
    X_RF = all_RF
    y_classify = all_labels
    y_location = all_positions
    print(f"Data reading completed! Audio data set shape: {X_audio.shape}, RF data set shape: {X_RF.shape}, Category label shape: {y_classify.shape}, Positioning label shape: {y_location.shape}")
    return X_audio,X_RF,y_classify,y_location
    
def post_process_data(
    X_audio,
    X_RF,
    y_classify,
    y_location,
    test_size = 0.2
):

    X_audio = np.transpose(X_audio, (0, 2, 1))
    X_RF = np.transpose(X_RF,(0,2,1))
    print(f"Data shape after transpose: Audio shape-{X_audio.shape}, RF shape-{X_RF.shape}")
    np.random.seed(42)
    
    indices = np.random.permutation(X_audio.shape[0])
    print(indices.shape)
    X_audio = X_audio[indices]
    X_RF = X_RF[indices]
    y_classify = y_classify[indices]
    y_location = y_location[indices]  
    
    
    X_train_val_audio,X_test_audio,y_train_val_classify,y_test_classify,X_train_val_RF,X_test_RF,y_train_val_location,y_test_location = train_test_split(X_audio,y_classify,X_RF,y_location,test_size=test_size,shuffle=True,random_state=random_state,stratify=y_classify)  

    train_mean = np.mean(X_train_val_audio, axis=(0,1))
    train_std = np.std(X_train_val_audio, axis=(0,1))
    print('Observe statistical feature dimensions')
    print(train_mean.shape)
    print(train_std.shape)
    X_train_val_audio = (X_train_val_audio - train_mean) / (train_std + 1e-6)
    X_test_audio = (X_test_audio - train_mean) / (train_std + 1e-6)
    print('Normalized data set shape:')
    print(X_train_val_audio.shape, y_train_val_classify.shape,X_train_val_RF.shape, y_train_val_location.shape)
    print(X_test_audio.shape, y_test_classify.shape, X_test_RF.shape,y_test_location.shape)
    
    return train_mean,train_std,X_train_val_audio,X_test_audio,y_train_val_classify,y_test_classify,X_train_val_RF,X_test_RF,y_train_val_location,y_test_location
    
X_audio,X_RF,y_classify,y_location = load_microphone_signal(data_folder,drone_types,fs_audio,N_audio,RF_suffixes,loca_suffixes,num_classes,mic_num)
test_size = 0.2
random_state = 42
train_mean,train_std,X_train_val_audio,X_test_audio,y_train_val_classify,y_test_classify,X_train_val_RF,X_test_RF,y_train_val_location,y_test_location = post_process_data(X_audio,X_RF,y_classify,y_location,test_size=test_size)
train_mean_tf = tf.constant(train_mean,dtype=tf.float32)
train_std_tf = tf.constant(train_std,dtype=tf.float32)
t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")


In [ ]:
# =========== The second step extracts the amplitude difference and time difference of the original signal for classification tasks =====================
def prepare_data_for_pinn(X):
    n_channels = X.shape[2]
    return [X[:,:,c:c+1] for c in range(n_channels)]

class TimeDelayFeatureLayer(tf.keras.layers.Layer):

    def __init__(self,fs_audio,distance_mm = 500.0,sound_speed=340.0,frame_length = 2048,frame_step = 1024,**kwargs):
        super().__init__(**kwargs)
        self.fs = fs_audio
        self.distance_m = distance_mm/1000.0
        self.sound_speed = sound_speed
        self.frame_length = frame_length
        self.frame_step = frame_step


    def build(self,input_shape):
        self.n_channels = input_shape[2]
        self.n_pairs = self.n_channels * (self.n_channels - 1) // 2
        
        max_delay_seconds = 2*self.distance_m / self.sound_speed
        self.max_delay_samples = int(round(max_delay_seconds * self.fs))

    def call(self,audio_data):
        frames = tf.signal.frame(audio_data,self.frame_length,self.frame_step,axis = 1)
        pair_feats_per_frame = []
        for ch1 in range(self.n_channels):
            for ch2 in range(ch1+1,self.n_channels):
                
                s1 = frames[:,:,:,ch1]
                s2 = frames[:,:,:,ch2]

                
                s1 = s1 - tf.reduce_mean(s1,axis=2,keepdims=True)
                s2 = s2 - tf.reduce_mean(s2,axis=2,keepdims=True)

                # FFT
                S1 = tf.signal.fft(tf.cast(s1,tf.complex64))
                S2 = tf.signal.fft(tf.cast(s2,tf.complex64))
                corr_fft = S1*tf.math.conj(S2)
                # IFFT
                corr = tf.math.real(tf.signal.ifft(corr_fft))
                center = self.frame_length//2
                mini = tf.maximum(center-self.max_delay_samples,0)
                maxi = tf.minimum(center+self.max_delay_samples+1,self.frame_length)
                corr_focus = corr[:,:,mini:maxi]
                
                idx = tf.argmax(corr_focus,axis=2,output_type=tf.int32)
                lag_value = tf.reduce_mean(tf.cast(idx,tf.float32),axis=1)/float(self.max_delay_samples)
                pair_feats_per_frame.append(lag_value[:,None])
        pair_feats = tf.concat(pair_feats_per_frame,axis=1)
        return pair_feats


class AmpDiffFeatureLayer(tf.keras.layers.Layer):
    def build(self,input_shape):
        self.n_channels = input_shape[2]
        self.n_pairs = self.n_channels * (self.n_channels-1)//2
    def call(self,audio_data):

        means = tf.reduce_mean(tf.abs(audio_data),axis=1)
        pair_feats = []
        
        for i in range(self.n_channels):
            for j in range(i+1,self.n_channels):
                amp_diff = means[:,i]-means[:,j]
                pair_feats.append(amp_diff[:,None])
        pair_feats = tf.concat(pair_feats,axis=1)
        return pair_feats


X_train_val_pinn = prepare_data_for_pinn(X_train_val_audio)
X_test_pinn = prepare_data_for_pinn(X_test_audio)

print('Train_val_set:',[x.shape for x in X_train_val_pinn],y_train_val_classify.shape)
print('Test_set:',[x.shape for x in X_test_pinn],y_test_classify.shape)


In [ ]:

# ======== The third step is to set the reconstruction part of the microphone array, the TDOA loss part, and the fluctuation PDE residual part.============

MIC_POSITIONS = np.array([
    [0.5,0,0],
    [0,0.5,0],
    [-0.5,0,0],
    [0,-0.5,0]
],dtype=np.float32)
DEFAULT_MIC_POSITIONS = tf.constant(MIC_POSITIONS,dtype=tf.float32)
PROPELLER_OFFSETS_Air3s = tf.constant([
    [-0.133,-0.163,0.053],
    [0.133,-0.163,0.053],
    [0.133,0.163,0.053],
    [-0.133,0.163,0.053]
],dtype = tf.float32)
PROPELLER_OFFSETS_Mavic2pro = tf.constant([
    [-0.161,-0.121,0.042],
    [0.161,-0.121,0.042],
    [0.161,0.121,0.042],
    [-0.161,0.121,0.042]
],dtype = tf.float32)
PROPELLER_OFFSETS_Mini3pro = tf.constant([
    [-0.126,-0.181,0.035],
    [0.126,-0.181,0.035],
    [0.126,0.181,0.035],
    [-0.126,0.181,0.035]
],dtype = tf.float32)

PROPELLER_OFFSETS_Mini5pro = tf.constant([
    [-0.152,-0.190,0.046],
    [0.152,-0.190,0.046],
    [0.152,0.190,0.046],
    [-0.152,0.190,0.046]
],dtype = tf.float32)

PROPELLER_OFFSETS_Mean = tf.constant([
    [-0.143,-0.164,0.044],
    [0.143,-0.164,0.044],
    [0.143,0.164,0.044],
    [-0.143,0.164,0.044]
],dtype = tf.float32)
DEFAULT_PROPELLER_OFFSETS = tf.stack([PROPELLER_OFFSETS_Mean,PROPELLER_OFFSETS_Mean,PROPELLER_OFFSETS_Mean,PROPELLER_OFFSETS_Mean]) 

def select_propeller_offsets(classify_result,DEFAULT_PROPELLER_OFFSETS):

    cls_prob_exp = tf.expand_dims(classify_result,axis=-1)
    cls_prob_exp = tf.expand_dims(cls_prob_exp,axis=-1)
    offsets_exp = tf.expand_dims(DEFAULT_PROPELLER_OFFSETS,axis=0)
    weighted_offsets = cls_prob_exp*offsets_exp
    return tf.reduce_sum(weighted_offsets,axis=1)

# ============================    Next, the signal reconstruction part, TDOA loss part and fluctuating PDE residual part. ===================
def get_alfa_tf(range_val,Tin,Hrin,Psin,f):
    
    T = Tin + 273.15
    To1 = 273.15
    To =293.15
    Pso=1.0
    Ps = Psin/29.9212598
    F = f/Ps
    PSat = (tf.pow(10.0,
        10.79586*(1-To1/T)
        - 5.02808 * tf.math.log(T/To1)/tf.math.log(10.0)
        + 1.50474e-4 * (1-tf.pow(10.0, -8.29692 * (T/To1-1)))
        + 0.42873e-3 * (tf.pow(10.0,4.76955 * (1-To1/T))-1)
        - 2.2195983))
    h = Hrin * PSat /Ps
    frO = (Ps/Pso)*(24+4.04e4*h*(0.02+h)/(0.391+h))
    frN = (Ps/Pso)*tf.sqrt(To/T)*(9+280*h*tf.exp(-4.17*(((To/T)**(1/3))-1)))
    
    alpha = ((Ps/Pso) * tf.square(F) *
            (1.84e-11 * tf.sqrt(T/To) 
            + (T/To)**-2.5*
              (0.01278 * tf.exp(-2239.1/T)/(frO+tf.square(F)/frO)+
               0.1068*tf.exp(-3352/T)/(frN+tf.square(F)/frN)))
            )
    
    alfa_db = 20*tf.math.log(tf.exp(alpha*range_val))/tf.math.log(10.0)
    return alfa_db


class AcousticSourceModel(tf.keras.layers.Layer):

    def __init__(self,mic_positions = DEFAULT_MIC_POSITIONS,train_mean_tf = train_mean_tf,train_std_tf = train_std_tf,Nh=5,c=340.0,fs_audio=fs_audio,N_audio=N_audio,**kwargs):
        super().__init__(**kwargs)
        self.mic_positions = tf.constant(mic_positions,dtype=tf.float32)
        self.train_mean_tf = tf.constant(train_mean_tf,dtype=tf.float32)
        self.train_std_tf = tf.constant(train_std_tf,dtype=tf.float32)
        self.Nh = Nh 
        self.c = c
        self.fs = fs_audio
        
        self.f0 = tf.Variable(80.0,dtype=tf.float32,constraint = lambda x: tf.clip_by_value(x,50.0,100.0))
        self.N = N_audio
        self.n_mics = mic_positions.shape[0]

    def normalize_signal(self,signal):
        
        signal = tf.transpose(signal,perm=[0,2,1])
        signal_norm = (signal - self.train_mean_tf)/(self.train_std_tf+1e-6)
        signal_norm = tf.transpose(signal_norm,perm=[0,2,1])
        return signal_norm   
        
    def unnormalize_signal(self,signal_normalized):
        
        signal_normalized = tf.transpose(signal_normalized,perm=[0,2,1])
        signal_normalized = signal_normalized *(self.train_std_tf+1e-6)+self.train_mean_tf
        signal_normalized = tf.transpose(signal_normalized,perm=[0,2,1])
        return signal_normalized
        
    def call(self,drone_pos,classify_result,return_normalized=True):

        batch_size = tf.shape(drone_pos)[0]
        mic_positions = self.mic_positions
        N = self.N
        dt = 1.0/self.fs
        n_mics = self.n_mics
        PROPELLER_OFFSETS = select_propeller_offsets(classify_result,DEFAULT_PROPELLER_OFFSETS)

        
        prop_positions = tf.expand_dims(drone_pos,1)+PROPELLER_OFFSETS #[batch,4,3]

        
        props_expand = tf.expand_dims(prop_positions,2)# [batch,4,1,3]
        mics_expand = tf.expand_dims(tf.expand_dims(mic_positions,0),0) # [1,1,n_mic,3]
        mic_prop_deltas = props_expand-mics_expand
        distances = tf.norm(mic_prop_deltas,axis=-1) # [batch,4,n_mic]
        delays = distances/self.c  # [batch,4,n_mic]
        t = tf.range(N,dtype=tf.float32)*dt
        t = tf.reshape(t,[1,1,1,N]) # [1,1,1,N]
        received = tf.zeros([batch_size,n_mics,N],dtype=tf.float32)
        Q_source = tf.zeros([batch_size,n_mics,N],dtype=tf.float32)
        Tin = 15.0
        Hrin = 0.4
        Psin = 30.0

        for n in range(1,self.Nh+1):
            cur_range=distances
            f_weak = (2.0*n-1.0)*self.f0
            f_strong = (2.0*n)*self.f0

            # ==== Weak odd harmonics ==== 
            alfa_db_weak = get_alfa_tf(cur_range,Tin,Hrin,Psin,f_weak)
            att_weak = tf.pow(10.0,-alfa_db_weak/20.0)
            phase_weak = 2*np.pi*f_weak*(t-tf.expand_dims(delays,-1)) # [B,4,n_mic,N]
            weak_signal = tf.reduce_sum(att_weak[...,None]*tf.cos(phase_weak)/(4.0*np.pi*tf.expand_dims(cur_range,-1)),axis=1)
            Q_weak = tf.reduce_sum(tf.cos(2*np.pi*f_weak*t),axis=1)

            # ==== Strong even harmonics ==== 
            alfa_db_strong = get_alfa_tf(cur_range,Tin,Hrin,Psin,f_strong)
            att_strong = tf.pow(10.0,-alfa_db_strong/20.0)
            phase_strong = 2*np.pi*f_strong*(t-tf.expand_dims(delays,-1)) # [B,4,n_mic,N]
            strong_signal = tf.reduce_sum(att_strong[...,None]*tf.cos(phase_strong)/(4.0*np.pi*tf.expand_dims(cur_range,-1)),axis=1)
            Q_strong = tf.reduce_sum(tf.cos(2*np.pi*f_strong*t),axis=1)

            received += weak_signal + strong_signal
            Q_source += Q_weak + Q_strong
            
        synth_signal = received
        
        if return_normalized:
            synth_signal_norm = self.normalize_signal(synth_signal)
            Q_source_norm = self.normalize_signal(Q_source)
            return tf.transpose(synth_signal_norm,[0,2,1]),tf.transpose(Q_source_norm,[0,2,1])
        else:
            synth_signal_phys = self.unnormalize_signal(synth_signal)
            Q_source_phys = self.unnormalize_signal(Q_source)
            return tf.transpose(synth_signal_phys,[0,2,1]),tf.transpose(Q_source_phys,[0,2,1])

    
# ==== TDOA Loss ====
class TDOAPhysicsLayer(tf.keras.layers.Layer):
    
    def __init__(self, mic_positions, c=340.0, sr=fs_audio,**kwargs):

        super().__init__(**kwargs)
        self.mic_pos = tf.constant(mic_positions, dtype=tf.float32)  # (4,3)
        self.c = float(c)
        self.sr = float(sr)
        self.n_mics = mic_positions.shape[0]
        
        self.pair_indices = np.array(list(combinations(range(self.n_mics),2)),dtype=np.int32)
        
    def _compute_theoretical_tdoa(self, pred_pos):

        dist = tf.norm(pred_pos[:, None, :] - self.mic_pos, axis=-1)  # (batch, n_mic)
        dist_1 = tf.gather(dist,self.pair_indices[:,0],axis=1)
        dist_2 = tf.gather(dist,self.pair_indices[:,1],axis=1)
        tdoa_pred = (dist_1-dist_2)/self.c  # (batch, n_pairs)
        return tdoa_pred
        
    def _compute_measured_tdoa(self, signals):

        signals_f = tf.cast(signals, tf.complex64)    # (batch, seq_len, n_mic)
        seq_len = tf.shape(signals_f)[1]
        half_len = seq_len // 2

        def gcc_phat_one_batch(batch_signals):  # batch_signals: (seq_len, n_mic)
            # shape: (n_pairs,)
            sigs = batch_signals    # (seq_len, n_mic)
    
            def pair_gcc(pair):
                i = tf.cast(pair[0], tf.int32)
                j = tf.cast(pair[1], tf.int32)
                sig_i = sigs[:, i]  # (seq_len,)
                sig_j = sigs[:, j]
                fft_i = tf.signal.fft(sig_i)
                fft_j = tf.signal.fft(sig_j)
                cross_power = fft_i * tf.math.conj(fft_j)
                denominator = tf.cast(tf.abs(cross_power) + 1e-8, tf.complex64)
                cross_corr = tf.signal.ifft(cross_power / denominator)     # (seq_len,)
                abs_corr = tf.math.abs(cross_corr)
                max_idx = tf.cast(tf.argmax(abs_corr), tf.int32)  
                delay = tf.math.floormod(max_idx + half_len, seq_len) - half_len  # int32
                tdoa = tf.cast(delay, tf.float32) / self.sr
                return tdoa   # scalar

            tdoas = tf.map_fn(pair_gcc, self.pair_indices, fn_output_signature=tf.float32)  # (n_pairs,)
            return tdoas  # (n_pairs,)

        
        tdoas_batch = tf.map_fn(gcc_phat_one_batch, signals_f, fn_output_signature=tf.float32)  # (batch, n_pairs)
        return tdoas_batch

    def call(self, pred_pos, measured_signals):
        tdoa_theory = self._compute_theoretical_tdoa(pred_pos)
        tdoa_meas = self._compute_measured_tdoa(measured_signals)
            
        
        return tdoa_theory - tdoa_meas
  
class WaveResidualCalculator(tf.keras.layers.Layer):    
    def __init__(self, c=340.0, sensitivity=12.6, source_width=0.05, **kwargs):        
        super().__init__(**kwargs)
        self.c = c  
        self.sensitivity = sensitivity  #12.6mV/Pa
        self.source_width = source_width 
    def call(self,p,coords,t,Q,pos_pred,classify_result):
        # 1. first convert both P and Q from the voltage domain to the sound pressure domain.
        P_pa = p/self.sensitivity
        Q_pa = Q/self.sensitivity
        # 2. calculate the time derivative
        dt = t[0,1]-t[0,0]
        P_tt = (P_pa[:,2:,:]-2*P_pa[:,1:-1,:]+P_pa[:,:-2,:])/(dt**2)
        # 3. Laplace approximation
        d = tf.constant(0.5,dtype=tf.float32)
        lapl_0 = (P_pa[:,1:-1,2]-2*P_pa[:,1:-1,0]+P_pa[:,1:-1,1])/(d**2)
        lapl_1 = (P_pa[:,1:-1,0]-2*P_pa[:,1:-1,1]+P_pa[:,1:-1,2])/(d**2)
        lapl_2 = (P_pa[:,1:-1,1]-2*P_pa[:,1:-1,2]+P_pa[:,1:-1,3])/(d**2)
        lapl_3 = (P_pa[:,1:-1,2]-2*P_pa[:,1:-1,3]+P_pa[:,1:-1,0])/(d**2)
        
        laplacian = tf.stack([lapl_0,lapl_1,lapl_2,lapl_3],axis=-1)
        
        PROPELLER_OFFSETS = select_propeller_offsets(classify_result,DEFAULT_PROPELLER_OFFSETS)
        
        prop_positions = tf.expand_dims(pos_pred,axis=1)+PROPELLER_OFFSETS
        # 4. Broadcast the distance between each propeller of each batch and all coordinates
        coords_expand = tf.expand_dims(coords,axis=1)
        prop_expand = tf.expand_dims(prop_positions,axis=2)
        dist = tf.norm(coords_expand-prop_expand,axis=-1)
        
        sigma = self.source_width
        norm_coef = 1.0/((2*np.pi*sigma**2)**(3/2))
        expo = tf.exp(-0.5*tf.square(dist/sigma))
        delta_approx = norm_coef*expo
        Q_time = Q_pa[:,1:-1,:]
        time_steps = tf.shape(Q_time)[1]
        delta_approx_tiled = tf.expand_dims(delta_approx,axis=1)
        delta_approx_tiled = tf.tile(delta_approx_tiled,[1,time_steps,1,1])
        Q_exp = tf.expand_dims(Q_time,axis=2)
        mic_contri = delta_approx_tiled*Q_exp
        Q_full = tf.reduce_sum(mic_contri,axis=3)
        wave_residual = laplacian-(1/self.c**2)*P_tt+Q_full
        return wave_residual


In [ ]:
# ============  Establish Network ==================
# ++++++++++++++++++  Recognition branch network  ++++++++++++++++
class ChannelProcessor(layers.Layer):
    
    def __init__(self,channel_id,fs_audio,N_audio,**kwargs):
        super().__init__(**kwargs)
        self.channel_id = channel_id
        self.fs = fs_audio
        self.N = N_audio
        
        self.conv = layers.Conv1D(16,kernel_size=64,strides=32,padding='same')
        self.act = layers.Activation('relu')
        self.bn = layers.BatchNormalization()
        self.pool = layers.MaxPooling1D(4)
        self.gap = layers.GlobalAveragePooling1D()
        self.fc = layers.Dense(8,activation='relu',name=f'channel_dense1_ch{channel_id}')

    def call(self,x):
        x = self.conv(x)
        x = self.act(x)
        x = self.bn(x)
        x = self.pool(x)
        x = self.gap(x)
        x = self.fc(x)
        return x
    def get_config(self):
        config = super().get_config().copy()
        config.update({'channel_id':self.channel_id,'fs':self.fs,'N':self.N})
        return config

class Audio_RF_Model(tf.keras.Model):
    
    def __init__(self,fs_audio,mic_num,num_classes,N_audio,physics_delay,physics_amp,physics_orth,input_shape_RF=(seq_length,2),name="classification_model"):
        super().__init__(name=name)
        self.mic_num = mic_num
        self.N = N_audio

        self.channel_processors = [
            ChannelProcessor(i+1,fs_audio,N_audio,name=f'channel_proc_{i+1}') for i in range(mic_num)
        ]
        self.audio_fusion = layers.Concatenate(name='audio_fusion')

        self.class_dense1=layers.Dense(32,activation='relu',name='class_audio_dense1')
        self.class_dense2 = layers.Dense(16,activation='relu',name='class_audio_dense2')
        # time difference
        self.audio_delay_dense = layers.Dense(16,activation='relu',name='audio_delay_dense')
        self.audio_out_delay = layers.Dense(6,activation=None,name='audio_output_delay')
        # amp difference
        self.audio_amp_dense = layers.Dense(16,activation='relu',name='audio_amp_dense')
        self.audio_out_amp = layers.Dense(6,activation=None,name='audio_output_amp')
        
        self.audio_phys_fusion = layers.Concatenate(name='audio_physics_fusion')
        self.audio_physics_dense = layers.Dense(16,activation='relu',name='audio_physics_dense')


        self.delay_phy_layer = TimeDelayFeatureLayer(fs_audio=fs_audio)
        self.amp_phy_layer = AmpDiffFeatureLayer()


        self.physics_weight_delay = tf.Variable(physics_delay,trainable=False,dtype=tf.float32,name='delay_loss_weight')
        self.physics_weight_amp = tf.Variable(physics_amp,trainable=False,dtype=tf.float32,name='amp_loss_weight')
        self.loss_delay_metrics = tf.keras.metrics.Mean(name='loss_delay')
        self.loss_amp_metrics = tf.keras.metrics.Mean(name='loss_amp')


        #+++++++++ RF ++++++++++++++
        self.input_shape_RF = input_shape_RF
        # self.permute = layers.Permute((2,1))
        self.RF_conv = layers.Conv1D(16,8,strides=8,padding='same',activation='relu',data_format='channels_last')
        self.RF_pool = layers.MaxPooling1D(pool_size=8,data_format='channels_last')
        self.RF_global_pool = layers.GlobalAveragePooling1D()
        self.RF_concat = layers.Concatenate()
        self.RF_dense1 = layers.Dense(16,activation = 'relu')
        self.RF_dense2 = layers.Dense(8,activation = 'relu')
        self.physics_ortho_weight = tf.Variable(physics_orth,trainable=False,dtype=tf.float32,name='physics_orth_weight')
        self.loss_orth_metrics = tf.keras.metrics.Mean(name='loss_orth')
        # +++++++++++++ Fusion +++++++++++++
        self.fusion_dense1 = layers.Dense(32,activation='relu')
        self.fusion_dense2 = layers.Dense(16,activation='relu')
        self.class_output = layers.Dense(num_classes,activation='softmax',name='final_out_cls')

    def call(self,inputs,training=None):
        inputs_audio = inputs[0:4]
        inputs_RF = inputs[4]

        audio_feats = []
        for i in range(self.mic_num):
            x = self.channel_processors[i](inputs_audio[i])
            audio_feats.append(x)
        audio_merged = self.audio_fusion(audio_feats)

        x_audio = self.class_dense1(audio_merged)
        x_audio = self.class_dense2(x_audio)

        audio_delay_pred = self.audio_out_delay(self.audio_delay_dense(audio_merged))
        audio_amp_pred = self.audio_out_amp(self.audio_amp_dense(audio_merged))

        audio_physics_cat = self.audio_phys_fusion([x_audio,audio_delay_pred,audio_amp_pred])
        audio_feat_32 = self.audio_physics_dense(audio_physics_cat)


        I_RF = inputs_RF[:,:,0:1]
        Q_RF = inputs_RF[:,:,1:2]
        conv_I = self.RF_conv(I_RF)
        conv_Q = self.RF_conv(Q_RF)
        pooled_I = self.RF_pool(conv_I)
        pooled_Q = self.RF_pool(conv_Q)
        global_I = self.RF_global_pool(pooled_I)
        global_Q = self.RF_global_pool(pooled_Q)
        merged_RF = self.RF_concat([global_I,global_Q])
        x_RF = self.RF_dense1(merged_RF)
        RF_feat_32 = self.RF_dense2(x_RF)

        # ======= Orth loss ==========
        batch_size = tf.shape(conv_I)[0]
        I_flat = tf.reshape(conv_I,[batch_size,-1])
        Q_flat = tf.reshape(conv_Q,[batch_size,-1])
        I_norm = tf.nn.l2_normalize(I_flat,axis = 1)
        Q_norm = tf.nn.l2_normalize(Q_flat,axis = 1)
        cos_sim = tf.reduce_mean(tf.maximum(0.0,tf.reduce_sum(I_norm*Q_norm,axis=1)))
        total_RF_loss = self.physics_ortho_weight*cos_sim
        # ====== Time and amp difference loss ==========
        audio_data = tf.concat(inputs_audio,axis=2)
        delay_true = self.delay_phy_layer(audio_data)
        amp_true = self.amp_phy_layer(audio_data)

        delay_loss = tf.reduce_mean(tf.square(audio_delay_pred-delay_true))
        amp_loss = tf.reduce_mean(tf.square(audio_amp_pred-amp_true))
        total_audio_loss = self.physics_weight_delay*delay_loss + self.physics_weight_amp*amp_loss
        self.add_loss(total_audio_loss)
        self.add_loss(total_RF_loss)

        self.loss_delay_metrics.update_state(delay_loss)
        self.loss_amp_metrics.update_state(amp_loss)
        self.loss_orth_metrics.update_state(cos_sim)

        fusion_feat = tf.concat([audio_feat_32,RF_feat_32],axis=-1)
        x = self.fusion_dense1(fusion_feat)
        x = self.fusion_dense2(x)
        out_cls = self.class_output(x)

        return {'output_cls':out_cls}
  

# +++++++++++++++++++++++++  Location branch network ++++++++++++++++++++++++++
class SqueezeExcite(layers.Layer):
    # SE
    def __init__(self,channels,ratio=8):
        super().__init__()
        self.avg = layers.GlobalAveragePooling1D()
        self.fc1 = layers.Dense(max(channels//ratio,4),activation='relu')
        self.fc2 = layers.Dense(channels,activation='sigmoid')
    def call(self,x):
        w = self.fc2(self.fc1(self.avg(x)))
        w = tf.expand_dims(w,1)
        return x*w

class Acoustic_loca(tf.keras.Model):
    def __init__(self,N_audio,mic_positions,train_mean_tf,train_std_tf,physics_wave,physics_tdoa,physics_signal,Nh=5,c=340.0,fs_audio=48000):
        super().__init__()
        self.signal_length = N_audio
        self.train_mean_tf = train_mean_tf
        self.train_std_tf  = train_std_tf
        self.mic_positions = tf.constant(mic_positions,dtype=tf.float32)
        n_mic = mic_positions.shape[0]
        
        self.acoustic_model = AcousticSourceModel(mic_positions,train_mean_tf,train_std_tf,Nh=Nh,c=c,fs_audio=fs_audio,N_audio=N_audio)
        self.tdoa_layer = TDOAPhysicsLayer(mic_positions,c=c,sr=fs_audio)
        self.wave_residual_calc = WaveResidualCalculator()
        
        self.physics_weight_wave_residual = tf.Variable(physics_wave,trainable=False,dtype=tf.float32,name="physics_weight_wave")
        self.physics_weight_tdoa_residual = tf.Variable(physics_tdoa,trainable=False,dtype=tf.float32,name="physics_weight_tdoa")
        self.physics_weight_signal_recap = tf.Variable(physics_signal,trainable = False,dtype=tf.float32,name="physics_weight_signal")

    
        self.conv1 = layers.Conv1D(32,15,strides=3,padding='same',data_format="channels_last",activation='relu')
        self.bn1 = layers.BatchNormalization()
        self.conv2 = layers.Conv1D(64,7,strides=2,padding='same',data_format = "channels_last",activation='relu')
        self.bn2 = layers.BatchNormalization()
        self.se1 = SqueezeExcite(64)
        self.conv3 = layers.Conv1D(128,5,strides=2,padding='same',data_format = "channels_last",activation='relu')
        self.bn3 = layers.BatchNormalization()
        self.se2 = SqueezeExcite(128)
        self.global_pool = layers.GlobalAveragePooling1D()
        
        self.dense1 = layers.Dense(128,activation='relu')
        self.dropout1 = layers.Dropout(0.2)
        self.dense2 = layers.Dense(64,activation='relu')
        self.dropout2 = layers.Dropout(0.2)
        self.dense3 = layers.Dense(32,activation='relu')
        self.position_layer = layers.Dense(3,name='position')

        self.loss_wave_metric = tf.keras.metrics.Mean(name='loss_wave')
        self.loss_tdoa_metric = tf.keras.metrics.Mean(name='loss_tdoa')
        self.loss_signal_metric = tf.keras.metrics.Mean(name='loss_signal')

    def call(self,inputs,training=None):

        mic_signals = inputs['mic_signals'] # [B,N,n_mic]
        time_vec = inputs['time_samples']
        classify_result = inputs['classify_result']
        batch_size = tf.shape(mic_signals)[0]
        expand_class = tf.expand_dims(classify_result,1)
        expand_class = tf.tile(expand_class,[1,tf.shape(mic_signals)[1],1])
        mic_signals_concat = tf.concat([mic_signals,expand_class],axis=-1)
        
        x = mic_signals_concat
        if x.shape[-1]<x.shape[1]:
            pass
        else:
            x = tf.transpose(x,perm=[0,2,1])
    
        x = self.conv1(x)
        x = self.bn1(x,training=training)
        
        x = self.conv2(x)
        x = self.bn2(x,training=training)
        x = self.se1(x)

        x = self.conv3(x)
        x = self.bn3(x,training = training)
        x = self.se2(x)

        x = self.global_pool(x)
        x = self.dense1(x)
        x = self.dense2(x)
        x = self.dense3(x)

        pos_pred = self.position_layer(x)

        synth_signals_phys,Q_source_phys = self.acoustic_model(pos_pred,classify_result,return_normalized=False)
   
        mic_signals_phys = mic_signals * (self.train_std_tf+1e-6) + self.train_mean_tf
        
        tdoa_residual = self.tdoa_layer(pos_pred,mic_signals_phys)
        
        mic_pos_expand = tf.tile(tf.expand_dims(self.mic_positions,axis=0),[batch_size,1,1])
        wave_residual = self.wave_residual_calc(synth_signals_phys,mic_pos_expand,time_vec,Q_source_phys,pos_pred,classify_result)
        
        wave_loss =  tf.reduce_mean(tf.square(wave_residual))
        tdoa_loss =  tf.reduce_mean(tf.square(tdoa_residual))
        signal_loss = tf.reduce_mean(tf.square(synth_signals_phys-mic_signals_phys))

        loss_wave =  self.physics_weight_wave_residual * wave_loss
        loss_tdoa =  self.physics_weight_tdoa_residual * tdoa_loss
        loss_signal = self.physics_weight_signal_recap * signal_loss
       
        self.add_loss(loss_wave)
        self.add_loss(loss_tdoa)
        self.add_loss(loss_signal)

        self.loss_wave_metric.update_state(wave_loss)
        self.loss_tdoa_metric.update_state(tdoa_loss)
        self.loss_signal_metric.update_state(signal_loss)

        
        return {'position':pos_pred}

# +++++++++++++++++ whole network ++++++++++++++++
def build_full_NN_model(fs_audio,mic_positions,train_mean_tf,train_std_tf,mic_num,num_classes,N_audio,seq_length,physics_orth,physics_delay,physics_amp,physics_wave,physics_tdoa,physics_signal):
    inputs_dict = {}
    for i in range(mic_num):
        inputs_dict[f'channel_{i+1}_input'] = keras.Input(shape=(N_audio,1),name=f'channel_{i+1}_input')
    inputs_dict['RF_input'] = keras.Input(shape=(seq_length,2),name='RF_input')
    inputs_dict['loc_input']=keras.Input(shape=(N_audio,mic_num),name="loc_input")
    inputs_dict['time_samples'] = keras.Input(shape=(N_audio,),name='time_samples')

    channel_inputs = [inputs_dict[f'channel_{i+1}_input'] for i in range(mic_num)]
    channel_inputs.append(inputs_dict['RF_input'])
    
    cls_model = Audio_RF_Model(fs_audio=fs_audio,mic_num=mic_num,num_classes=num_classes,N_audio=N_audio,physics_delay=physics_delay,physics_amp=physics_amp,physics_orth=physics_orth,input_shape_RF=(2,seq_length))
    out_cls = cls_model(channel_inputs)
    
    loc_inputs = {
        "mic_signals":inputs_dict['loc_input'],
        "time_samples": inputs_dict['time_samples'],
        "classify_result":out_cls['output_cls']
    }
    acoustic_network = Acoustic_loca(
        N_audio = N_audio,
        mic_positions = mic_positions,
        train_mean_tf = train_mean_tf,
        train_std_tf = train_std_tf,
        physics_wave = physics_wave,
        physics_tdoa = physics_tdoa,
        physics_signal = physics_signal
    )
    pos_pred = acoustic_network(loc_inputs)
    output_cls = Lambda(lambda x: x,name="output_cls")(out_cls['output_cls'])
    final_pos_pred = Lambda(lambda x: x, name="pos_pred")(pos_pred['position'])
    
    model = keras.Model(
        inputs = inputs_dict,
        outputs = {"output_cls":output_cls,"pos_pred":final_pos_pred}
    )
    
    model.cls_model = cls_model
    model.physics_weight_wave_residual = acoustic_network.physics_weight_wave_residual
    model.physics_weight_tdoa_residual = acoustic_network.physics_weight_tdoa_residual
    model.physics_weight_signal_recap = acoustic_network.physics_weight_signal_recap
    model.physics_weight_delay = cls_model.physics_weight_delay
    model.physics_weight_amp = cls_model.physics_weight_amp
    model.physics_ortho_weight = cls_model.physics_ortho_weight
    return model

def make_fusion_dataset(X_RF,X_PINN,X_loc,t_samples,y_classify,y_location,batch_size,shuffle=True):
    N = X_loc.shape[0]
    inputs = {}
    mic_num = len(X_PINN)
    for i in range(mic_num):
        inputs[f'channel_{i+1}_input']=X_PINN[i].astype(np.float32)
    inputs["RF_input"] = X_RF
    inputs["loc_input"] = X_loc.astype(np.float32)
    # time_sample
    if t_samples is not None:
        t_s = np.tile(t_samples,(N,1)).astype(np.float32)
        inputs["time_samples"] = t_s

    targets = {
        "output_cls": y_classify.astype(np.float32),
        "pos_pred": y_location.astype(np.float32)
    }

    ds = tf.data.Dataset.from_tensor_slices((inputs,targets))
    if shuffle:
        ds = ds.shuffle(N)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    channel_inputs = [inputs[f'channel_{i+1}_input'] for i in range(mic_num)]
    channel_inputs.append(inputs['RF_input'])

    return ds,channel_inputs

def euclidean_distance_loss(y_true,y_pred):
    return tf.reduce_mean(tf.norm(y_pred-y_true,axis=-1))


In [7]:
def save_history_to_csv(history_data, filename,save_dir):
    with open(os.path.join(save_dir, filename), "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(np.mat(history_data).transpose().tolist())

def find_layer(model,layer_type):
    for layer in model.layers:
        if isinstance(layer,layer_type):
            return layer
        if hasattr(layer,'layers'):
            found = find_layer(layer,layer_type)
            if found is not None:
                return found
    return None

class LossHistory(tf.keras.callbacks.Callback):
    def on_train_begin(self,logs=None):
        self.losses = []
        self.wave_losses = []
        self.tdoa_losses = []
        self.signal_losses = []
        self.delay_losses = []
        self.amp_losses = []
        self.orth_losses = []

    def on_epoch_end(self,epoch,logs=None):
        self.losses.append(logs.get('loss'))
        acoustic_layer = find_layer(self.model,Acoustic_loca)
        assert acoustic_layer is not None,"Can't find Acoustic_loca layer"
        Audio_RF_layer = find_layer(self.model,Audio_RF_Model)
        assert Audio_RF_layer is not None,"Can't find Audio_RF layer"
        self.wave_losses.append(acoustic_layer.loss_wave_metric.result().numpy())
        self.tdoa_losses.append(acoustic_layer.loss_tdoa_metric.result().numpy())
        self.signal_losses.append(acoustic_layer.loss_signal_metric.result().numpy())

        self.delay_losses.append(Audio_RF_layer.loss_delay_metrics.result().numpy())
        self.amp_losses.append(Audio_RF_layer.loss_amp_metrics.result().numpy())
        self.orth_losses.append(Audio_RF_layer.loss_orth_metrics.result().numpy())

        if epoch % 200 == 0 or epoch == 999:
            print(f"Epoch {epoch},loss={logs.get('loss'):.4f},val_loss = {logs.get('val_loss'):.4f}")
            print("Learning rate:",self.model.optimizer.learning_rate.numpy())
            print(f"Wave_loss = {self.wave_losses[-1]},"
                f"TDOA_loss = {self.tdoa_losses[-1]},"
                f"Signal_loss = {self.signal_losses[-1]},"
                f"Delay_loss = {self.delay_losses[-1]},"
                f"Amp_loss = {self.amp_losses[-1]},"
                f"Orth_loss = {self.orth_losses[-1]}")

In [ ]:
def save_result_fold(save_dir,history,history_loss,predictions,y_val_classify,y_val_location,main_model,compile_metrics_value):

    train_loss = history.history['loss']
    valid_loss = history.history['val_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(train_loss, 'b', label='Training loss')
    plt.plot(valid_loss, 'r', label='Validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Loss.png'))
    plt.close()

    classification_model_accuracy = history.history['output_cls_accuracy']
    val_classification_model_accuracy = history.history['val_output_cls_accuracy']
    plt.figure(figsize=(10, 6))
    plt.plot(classification_model_accuracy, 'b', label='Training classification_model_accuracy')
    plt.plot(val_classification_model_accuracy, 'r', label='Validation classification_model_accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('classification_model_accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'classification_model_accuracy.png'))
    plt.close()

    pos_pred_euclidean_distance_loss = history.history['pos_pred_euclidean_distance_loss']
    val_pos_pred_euclidean_distance_loss = history.history['val_pos_pred_euclidean_distance_loss']
    plt.figure(figsize=(10, 6))
    plt.plot(pos_pred_euclidean_distance_loss, 'b', label='Training pos_pred_euclidean_distance_loss')
    plt.plot(val_pos_pred_euclidean_distance_loss, 'r', label='Validation pos_pred_euclidean_distance_loss')
    plt.xlabel('Epochs')
    plt.ylabel('pos_pred_euclidean_distance_loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'pos_pred_euclidean_distance_loss.png'))
    plt.close()

    
    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.wave_losses, 'b', label='Wave loss')
    plt.xlabel('Epochs')
    plt.ylabel('Wave Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Wave Loss.png'))
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.tdoa_losses, 'b', label='TDOA loss')
    plt.xlabel('Epochs')
    plt.ylabel('TDOA Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'TDOA Loss.png'))
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.signal_losses, 'b', label='Signal loss')
    plt.xlabel('Epochs')
    plt.ylabel('Signal Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Signal Loss.png'))
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.delay_losses, 'b', label='Delay loss')
    plt.xlabel('Epochs')
    plt.ylabel('Delay Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Delay Loss.png'))
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.amp_losses, 'b', label='Amp loss')
    plt.xlabel('Epochs')
    plt.ylabel('Amp Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Amp Loss.png'))
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(history_loss.orth_losses, 'b', label='Orth loss')
    plt.xlabel('Epochs')
    plt.ylabel('Orth Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Orth Loss.png'))
    plt.close()
    
    
    save_history_to_csv(history_loss.wave_losses, 'wave_loss.csv',save_dir)
    save_history_to_csv(history_loss.tdoa_losses, 'tdoa_loss.csv',save_dir)
    save_history_to_csv(history_loss.signal_losses, 'signal_loss.csv',save_dir)
    save_history_to_csv(history_loss.delay_losses, 'delay_loss.csv',save_dir)
    save_history_to_csv(history_loss.amp_losses, 'amp_loss.csv',save_dir)
    save_history_to_csv(history_loss.orth_losses, 'orth_loss.csv',save_dir)

    save_history_to_csv(history.history['loss'], 'train_loss.csv',save_dir)
    save_history_to_csv(history.history['val_loss'], 'val_loss.csv',save_dir)

    save_history_to_csv(history.history['output_cls_accuracy'], 'Class_Accuracy.csv',save_dir)
    save_history_to_csv(history.history['val_output_cls_accuracy'], 'Val_Class_Accuracy.csv',save_dir)

    save_history_to_csv(history.history['pos_pred_euclidean_distance_loss'], 'Loca_APE.csv',save_dir)
    save_history_to_csv(history.history['val_pos_pred_euclidean_distance_loss'], 'Val_Loca_APE.csv',save_dir)


    y_pred_classes_prob = predictions['output_cls']
    y_pred_classes = np.argmax(y_pred_classes_prob, axis=1)
    y_true_classes = np.argmax(y_val_classify, axis=1)
    y_pred_pos = predictions['pos_pred']
    errors = np.linalg.norm(y_pred_pos - y_val_location, axis=1)
    save_history_to_csv(errors, 'test_errors.csv',save_dir)
    
    APE, rmse = np.mean(errors), np.sqrt(np.mean(errors**2))
    print(f'APE: {APE:.3f}m, RMSE: {rmse:.3f}m, Max: {np.max(errors):.3f}m, Min: {np.min(errors):.3f}m')
    plt.figure(figsize=(10,6))
    plt.plot(errors, label='Position Error')
    plt.xlabel('Sample index')
    plt.ylabel('Error (m)')
    plt.legend()
    plt.title('Test Sample Position Error')
    plt.grid(True)
    plt.savefig(os.path.join(save_dir, 'Test_position_error.png'))
    plt.close()

    conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)
    drone_types = ['Air3S','Mavic2pro','Mini3pro','Mini5pro']
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=drone_types, yticklabels=drone_types)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Confusion Matrix')
    plt.savefig(os.path.join(save_dir, 'Confusion_Matrix.png'))
    plt.close()

    classification_rep = classification_report(y_true_classes, y_pred_classes, target_names=drone_types)
    print("Classification_Report:\n",classification_rep)
    with open(os.path.join(save_dir, 'Classification_Report.txt'),"w") as f:
        f.write(classification_rep)
    
    physics_orth = main_model.physics_ortho_weight.numpy()
    physics_wave = main_model.physics_weight_wave_residual.numpy()
    physics_tdoa = main_model.physics_weight_tdoa_residual.numpy()
    physics_signal = main_model.physics_weight_signal_recap.numpy()
    physics_delay = main_model.physics_weight_delay.numpy()
    physics_amp = main_model.physics_weight_amp.numpy()
    
    pd.DataFrame({'cls':[compile_metrics_value],'mae':[APE],'rmse':[rmse],'physics_orth':[physics_orth],'physics_wave':[physics_wave],'physics_tdoa':[physics_tdoa],'physics_signal':[physics_signal],'physics_delay':[physics_delay],'physics_amp':[physics_amp]}).to_csv(os.path.join(save_dir,'test_position_error.csv'))
    print("Save")

    main_model.save(save_dir + '/my_main_model',save_format = 'tf')
    filename = 'main_model_summary.txt'
    file_path = os.path.join(save_dir,filename)
    with open(file_path,'w') as f:
        main_model.summary(print_fn = lambda x: f.write(x+'\n'))

    return compile_metrics_value,APE


In [ ]:
def get_model(physics_orth,physics_delay,physics_amp,physics_wave,physics_tdoa,physics_signal):
    tf.keras.backend.clear_session()
    model = build_full_NN_model(fs_audio=fs_audio,mic_positions=MIC_POSITIONS,train_mean_tf=train_mean_tf,train_std_tf=train_std_tf,mic_num=4,num_classes=4,N_audio=N_audio,seq_length=seq_length,
                                physics_orth=physics_orth,physics_delay=physics_delay,physics_amp=physics_amp,physics_wave=physics_wave,physics_tdoa=physics_tdoa,physics_signal=physics_signal)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            'output_cls':tf.keras.losses.CategoricalCrossentropy(),
            'pos_pred':euclidean_distance_loss
        },
        loss_weights= {
            'output_cls':1.0,
            'pos_pred':1.0
        },
        metrics={
            'output_cls':['accuracy'],
            'pos_pred':[euclidean_distance_loss]
        }
    )
    return model
def train_with_history(model, X_train_dict, X_val_dict, epochs, batch_size, verbose=1):
    history_loss = LossHistory()
    result = model.fit(
        X_train_dict, validation_data=X_val_dict,
        epochs=epochs, batch_size=batch_size,
        callbacks=[history_loss], verbose=verbose
    )
    return result, history_loss


In [ ]:
physics_orth = 0.0
physics_delay = 0.0
physics_amp = 0.0
physics_wave = 0.0
physics_tdoa = 0.0
physics_signal = 0.0
nb_epoch = 1000
batch_size = 64
n_splits = 10


t0 = time.time()
loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
base_path = f"Mydata_Audio_RF_Fusion/Batchsize_{batch_size}_Nbepoch_{nb_epoch}/{loca}/"
os.makedirs(base_path,exist_ok = True)

y_trainval_cls = np.argmax(y_train_val_classify,axis=1)
print('y_trainval_cls former 20 items:',y_trainval_cls[:20])
print('Category distribution:',np.bincount(y_trainval_cls))

skf = StratifiedKFold(n_splits = n_splits,shuffle=True,random_state=random_state)
acc_list = []
ape_list = []
train_mi_list,val_mi_list = [],[]
print(f"\n=== {n_splits} fold cross-valdation  ====")

for fold,(train_idx,val_idx) in enumerate(skf.split(X_train_val_audio,y_trainval_cls),1):

    print(f"\n==== Fold {fold}/{n_splits} ====")
    fold_loca = str(time.strftime('%Y-%m-%d-%H-%M-%S'))
    fold_path = f"{fold_loca}-fold{fold}"
    fold_dir = os.path.join(base_path,fold_path)
    os.makedirs(fold_dir,exist_ok = True)


    X_train_pinn = [ch[train_idx] for ch in X_train_val_pinn]   # list of arrays
    X_val_pinn   = [ch[val_idx]   for ch in X_train_val_pinn] 
    X_train_audio = X_train_val_audio[train_idx]
    X_val_audio = X_train_val_audio[val_idx]
    X_train_RF = X_train_val_RF[train_idx]
    X_val_RF = X_train_val_RF[val_idx]

    y_train_classify = y_train_val_classify[train_idx]
    y_val_classify = y_train_val_classify[val_idx]
    y_train_location = y_train_val_location[train_idx]
    y_val_location = y_train_val_location[val_idx]

    print(f"Train size:{X_train_audio.shape[0]},Val size:{X_val_audio.shape[0]}")
    
    main_model = get_model(physics_orth,physics_delay,physics_amp,physics_wave,physics_tdoa,physics_signal)
    t_samples = np.arange(0,N_audio)*dt_audio
    X_train_dict,_ = make_fusion_dataset(X_train_RF,X_train_pinn,X_train_audio,t_samples,y_train_classify,y_train_location,batch_size=batch_size)
    X_val_dict,_ = make_fusion_dataset(X_val_RF,X_val_pinn,X_val_audio,t_samples,y_val_classify,y_val_location,batch_size=batch_size,shuffle=False)

    history,history_loss = train_with_history(main_model,X_train_dict,X_val_dict,nb_epoch,batch_size,verbose=0)
    
    results = main_model.evaluate(X_val_dict,verbose=0)
    
    metrics_dict = dict(zip(main_model.metrics_names, results))
    compile_metrics_value = metrics_dict['output_cls_accuracy']

    predictions = main_model.predict(X_val_dict)
    
    temp_acc,temp_ape = save_result_fold(fold_dir,history,history_loss,predictions,y_val_classify,y_val_location,main_model,compile_metrics_value)
    acc_list.append(temp_acc)
    ape_list.append(temp_ape)


    msg = (f"Fold {fold} Acc: {temp_acc:.4f} APE: {temp_ape:.4f}\n")
    with open(os.path.join(fold_dir, "fold_result.txt"), "w", encoding="utf-8") as f:
        f.write(msg)
      
t1=time.time()
elapsed_time = t1-t0
print(f"Time: {elapsed_time:.4f} s")

In [ ]:

print("\n==== ALL-fold Results ====")
acc_mean, acc_std = np.mean(acc_list), np.std(acc_list)
ape_mean, ape_std = np.mean(ape_list), np.std(ape_list)

print(f"Acc: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"APE: {ape_mean:.4f} ± {ape_std:.4f}")


summary_msg = (
    "==== ALL-fold Results ====\n"
    f"Acc: {acc_mean:.4f} ± {acc_std:.4f}\n"
    f"Ape: {ape_mean:.4f} ± {ape_std:.4f}"
)
summary_path = os.path.join(base_path, "results_summary.txt")


with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg)

print(f"Save to: {summary_path}")


In [ ]:
# ======================= 2. Retrain the final model and evaluate on the test set =====================
print("\nRetrain the final model and evaluate on the test set")
t_samples = np.arange(0,N_audio)*dt_audio
final_model = get_model(physics_orth,physics_delay,physics_amp,physics_wave,physics_tdoa,physics_signal)
X_train_val_dict,_ = make_fusion_dataset(X_train_val_RF,X_train_val_pinn,X_train_val_audio,t_samples,y_train_val_classify,y_train_val_location,batch_size=batch_size)
X_test_dict,_ = make_fusion_dataset(X_test_RF,X_test_pinn,X_test_audio,t_samples,y_test_classify,y_test_location,batch_size=batch_size,shuffle=False)

final_history, final_history_loss = train_with_history(final_model, X_train_val_dict, X_train_val_dict,nb_epoch, batch_size, verbose=0)
test_result = final_model.evaluate(X_test_dict, verbose=0)
test_metrics_dict = dict(zip(final_model.metrics_names, test_result))
test_compile_metrics_value = test_metrics_dict['output_cls_accuracy']
test_dir = os.path.join(base_path, "test")
os.makedirs(test_dir, exist_ok=True)
test_acc,test_ape = save_result_fold(test_dir, final_history, final_history_loss, final_model.predict(X_test_dict), y_test_classify, y_test_location,final_model,test_compile_metrics_value)

summary_msg_test = (
    "==== Test Results After ALL Fold ====\n"
    f"Test set Acc: {test_acc:.4f}\n"
    f"Test set Ape: {test_ape:.4f}"
)
summary_path = os.path.join(test_dir, "test_set_results_summary.txt")

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_msg_test)

print(f"Save to: {summary_path}")
